In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Cluster Heartbeat - Model Training\n",
    "\n",
    "## Training Fingerprint Autoencoder and Anomaly Detector\n",
    "\n",
    "This notebook demonstrates the training process for the fingerprint autoencoder and anomaly detector."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "from pathlib import Path\n",
    "\n",
    "sys.path.insert(0, str(Path.cwd().parent))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.config import load_config\n",
    "from src.data.ingestion import DataIngestion\n",
    "from src.features.extractor import FeatureExtractor\n",
    "from src.features.normalizer import FeatureNormalizer\n",
    "from src.models.fingerprint import FingerprintTrainer\n",
    "from src.models.anomaly import AnomalyDetector\n",
    "from src.training.train import train"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load configuration\n",
    "config = load_config('../config/config.yaml')\n",
    "print(\"Configuration loaded successfully!\")\n",
    "print(f\"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}\")\n",
    "print(f\"GPU available: {torch.cuda.is_available()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load and prepare data\n",
    "ingestion = DataIngestion(config)\n",
    "df = ingestion.load_data(source='synthetic')\n",
    "\n",
    "extractor = FeatureExtractor(config)\n",
    "feature_matrix, _ = extractor.extract_all_features(df)\n",
    "\n",
    "normalizer = FeatureNormalizer(config)\n",
    "features_norm = normalizer.fit_transform(feature_matrix)\n",
    "\n",
    "print(f\"Feature matrix shape: {features_norm.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Split data\n",
    "from sklearn.model_selection import train_test_split\n",
    "\n",
    "X_train, X_temp = train_test_split(features_norm, test_size=0.3, random_state=42)\n",
    "X_val, X_test = train_test_split(X_temp, test_size=0.5, random_state=42)\n",
    "\n",
    "print(f\"Training samples: {X_train.shape[0]}\")\n",
    "print(f\"Validation samples: {X_val.shape[0]}\")\n",
    "print(f\"Test samples: {X_test.shape[0]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize fingerprint trainer\n",
    "fingerprint_trainer = FingerprintTrainer(config)\n",
    "fingerprint_trainer.build_model(X_train.shape[1])\n",
    "\n",
    "print(f\"Model built: {fingerprint_trainer.model is not None}\")\n",
    "print(f\"Parameter count: {fingerprint_trainer.model.get_parameter_count():,}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train fingerprint model\n",
    "history = fingerprint_trainer.train(X_train, X_val)\n",
    "\n",
    "# Plot training history\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Loss curve\n",
    "axes[0].plot(history['train_losses'], label='Train Loss', linewidth=2)\n",
    "axes[0].plot(history['val_losses'], label='Validation Loss', linewidth=2)\n",
    "axes[0].axvline(x=history['best_epoch'], color='red', linestyle='--', \n",
    "                label=f'Best Epoch ({history[\"best_epoch\"]})')\n",
    "axes[0].set_xlabel('Epoch')\n",
    "axes[0].set_ylabel('Loss')\n",
    "axes[0].set_title('Training History')\n",
    "axes[0].legend()\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "# Loss distribution\n",
    "axes[1].hist(history['train_losses'], bins=30, alpha=0.7, label='Train', edgecolor='black')\n",
    "axes[1].hist(history['val_losses'], bins=30, alpha=0.7, label='Validation', edgecolor='black')\n",
    "axes[1].set_xlabel('Loss')\n",
    "axes[1].set_ylabel('Frequency')\n",
    "axes[1].set_title('Loss Distribution')\n",
    "axes[1].legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate fingerprints\n",
    "train_fingerprints = fingerprint_trainer.generate_fingerprints(X_train)\n",
    "val_fingerprints = fingerprint_trainer.generate_fingerprints(X_val)\n",
    "test_fingerprints = fingerprint_trainer.generate_fingerprints(X_test)\n",
    "\n",
    "print(f\"Train fingerprints: {train_fingerprints.shape}\")\n",
    "print(f\"Val fingerprints: {val_fingerprints.shape}\")\n",
    "print(f\"Test fingerprints: {test_fingerprints.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compute reconstruction errors\n",
    "train_errors = fingerprint_trainer.compute_reconstruction_error(X_train)\n",
    "val_errors = fingerprint_trainer.compute_reconstruction_error(X_val)\n",
    "test_errors = fingerprint_trainer.compute_reconstruction_error(X_test)\n",
    "\n",
    "# Visualize reconstruction errors\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Error distribution\n",
    "axes[0].hist(train_errors, bins=30, alpha=0.7, label='Train', edgecolor='black')\n",
    "axes[0].hist(val_errors, bins=30, alpha=0.7, label='Validation', edgecolor='black')\n",
    "axes[0].hist(test_errors, bins=30, alpha=0.7, label='Test', edgecolor='black')\n",
    "axes[0].set_xlabel('Reconstruction Error')\n",
    "axes[0].set_ylabel('Frequency')\n",
    "axes[0].set_title('Reconstruction Error Distribution')\n",
    "axes[0].legend()\n",
    "\n",
    "# Error statistics\n",
    "stats = {\n",
    "    'Train': [np.mean(train_errors), np.std(train_errors)],\n",
    "    'Val': [np.mean(val_errors), np.std(val_errors)],\n",
    "    'Test': [np.mean(test_errors), np.std(test_errors)]\n",
    "}\n",
    "\n",
    "x = np.arange(len(stats))\n",
    "width = 0.35\n",
    "\n",
    "axes[1].bar(x - width/2, [s[0] for s in stats.values()], width, label='Mean', alpha=0.7, edgecolor='black')\n",
    "axes[1].bar(x + width/2, [s[1] for s in stats.values()], width, label='Std', alpha=0.7, edgecolor='black')\n",
    "axes[1].set_xticks(x)\n",
    "axes[1].set_xticklabels(stats.keys())\n",
    "axes[1].set_ylabel('Error')\n",
    "axes[1].set_title('Reconstruction Error Statistics')\n",
    "axes[1].legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train anomaly detector\n",
    "anomaly_detector = AnomalyDetector(config)\n",
    "anomaly_detector.fit(train_fingerprints, train_errors)\n",
    "\n",
    "print(f\"Anomaly detector fitted: {anomaly_detector.is_fitted}\")\n",
    "print(f\"Anomaly threshold: {anomaly_detector.threshold:.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate on test set\n",
    "test_results = anomaly_detector.predict(test_fingerprints, test_errors)\n",
    "\n",
    "print(\"=\"*60)\n",
    "print(\"ANOMALY DETECTION EVALUATION\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\nTotal test samples: {len(test_results['scores'])}\")\n",
    "print(f\"Anomalies detected: {sum(test_results['predictions'])}\")\n",
    "print(f\"Anomaly rate: {sum(test_results['predictions'])/len(test_results['scores']):.2%}\")\n",
    "print(f\"Threshold: {test_results['threshold']:.4f}\")\n",
    "print(f\"Average score: {np.mean(test_results['scores']):.4f}\")\n",
    "print(f\"Max score: {np.max(test_results['scores']):.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize anomaly detection results\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Score distribution\n",
    "scores = np.array(test_results['scores'])\n",
    "predictions = np.array(test_results['predictions'])\n",
    "\n",
    "normal_scores = scores[predictions == 0]\n",
    "anomaly_scores = scores[predictions == 1]\n",
    "\n",
    "axes[0].hist(normal_scores, bins=20, alpha=0.7, label='Normal', edgecolor='black')\n",
    "axes[0].hist(anomaly_scores, bins=20, alpha=0.7, label='Anomaly', edgecolor='black')\n",
    "axes[0].axvline(x=test_results['threshold'], color='red', linestyle='--', \n",
    "                label=f'Threshold ({test_results[\"threshold\"]:.3f})')\n",
    "axes[0].set_xlabel('Anomaly Score')\n",
    "axes[0].set_ylabel('Frequency')\n",
    "axes[0].set_title('Anomaly Score Distribution')\n",
    "axes[0].legend()\n",
    "\n",
    "# Confusion matrix style\n",
    "categories = ['Normal', 'Anomaly']\n",
    "counts = [len(normal_scores), len(anomaly_scores)]\n",
    "\n",
    "axes[1].bar(categories, counts, color=['green', 'red'], alpha=0.7, edgecolor='black')\n",
    "axes[1].set_ylabel('Count')\n",
    "axes[1].set_title(f'Detection Results (Anomaly Rate: {sum(predictions)/len(predictions):.1%})')\n",
    "\n",
    "# Add labels on bars\n",
    "for i, count in enumerate(counts):\n",
    "    axes[1].text(i, count + 0.1, str(count), ha='center', va='bottom')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save models\n",
    "import pickle\n",
    "from pathlib import Path\n",
    "\n",
    "model_dir = Path('../models_checkpoints')\n",
    "model_dir.mkdir(exist_ok=True)\n",
    "\n",
    "# Save fingerprint model\n",
    "fingerprint_trainer.save_checkpoint(str(model_dir / 'fingerprint_model.pt'))\n",
    "\n",
    "# Save anomaly detector\n",
    "anomaly_detector.save(str(model_dir / 'anomaly_detector.pkl'))\n",
    "\n",
    "# Save normalizer\n",
    "normalizer.save(str(model_dir / 'normalizer.pkl'))\n",
    "\n",
    "print(f\"\\n✅ Models saved to: {model_dir}\")\n",
    "print(f\"   - fingerprint_model.pt\")\n",
    "print(f\"   - anomaly_detector.pkl\")\n",
    "print(f\"   - normalizer.pkl\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary\n",
    "print(\"=\"*60)\n",
    "print(\"MODEL TRAINING SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\n✅ Fingerprint Autoencoder:\")\n",
    "print(f\"   - Input dimension: {X_train.shape[1]}\")\n",
    "print(f\"   - Latent dimension: {config['model']['fingerprint']['latent_dim']}\")\n",
    "print(f\"   - Best validation loss: {history['best_val_loss']:.6f}\")\n",
    "print(f\"   - Best epoch: {history['best_epoch']}\")\n",
    "print(f\"   - Training time: {history['training_time']:.2f}s\")\n",
    "\n",
    "print(f\"\\n✅ Anomaly Detector:\")\n",
    "print(f\"   - Threshold: {anomaly_detector.threshold:.4f}\")\n",
    "print(f\"   - Test anomaly rate: {sum(test_results['predictions'])/len(test_results['scores']):.2%}\")\n",
    "\n",
    "print(\"\\n📊 Models ready for inference!\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}